# 03 — Análise Estatística e Geração de Tabelas/Figuras para o TCC

Este notebook lê exclusivamente os arquivos `metrics.json`, `history.csv` e `test_preds.npz` gerados pelos experimentos concluídos em `results/runs/` e produz:
- Média, desvio padrão e IC 95% por bootstrap sobre as 10 seeds;
- Shapiro-Wilk por grupo;
- Kruskal-Wallis global entre modelos;
- Mann-Whitney U par a par com correção de Holm e Cliff's delta (destacando os contrastes planejados);
- Tabelas `.csv`/`.tex` em `results/tables/`, figuras `.pdf`/`.png` em `results/figures/` e `results/summary.md`.


In [ ]:
import sys
from pathlib import Path

src_path = str((Path.cwd().parent / "src").resolve()) if (Path.cwd().parent / "src").exists() else str((Path.cwd() / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from tcc.config import load_config
from tcc.reporting import generate_full_report

EXPERIMENT_NAME = "main"  # Altere para "dry_run" ou outra ablação se desejar
cfg = load_config()
report = generate_full_report(cfg, experiment_name=EXPERIMENT_NAME)
print("Status:", report["status"], "| Runs carregados:", report["n_runs"])
print("Relatório salvo em:", report["summary_md"])

In [ ]:
# === Visualizar resumo e contrastes planejados ===
if report["status"] == "ok":
    display(report["summary_df"])
    if not report["kruskal_df"].empty:
        display(report["kruskal_df"])
    if not report["pairwise_df"].empty:
        contrasts = report["pairwise_df"][report["pairwise_df"]["contraste_planejado"] != ""]
        display(contrasts)

In [ ]:
# === Teste de McNemar opcional entre dois modelos usando test_preds.npz ===
from tcc.stats import mcnemar_test_from_preds

# Exemplo (preencha caminhos reais após concluir a grade):
# res = mcnemar_test_from_preds(
#     Path(cfg["paths"]["runs_dir"]) / "main/roberta-base_len512_seed42/test_preds.npz",
#     Path(cfg["paths"]["runs_dir"]) / "main/codebert-base_len512_seed42/test_preds.npz",
# )
# print(res)